In [1]:
import sys
from pathlib import Path
import pandas as pd

# --- Setup Project Paths ---
# This ensures the notebook can find the 'src' directory
# Assumes the notebook is in 'notebooks/' and src is a sibling directory
# /home/mouneer_alrayes/graphage13-sorption/
#   ├── notebooks/
#   └── src/
#
# Get the project root directory
project_root = Path.cwd().parent 
# Add the 'src' directory to the Python path
sys.path.append(str(project_root / 'src'))

# Define the data directory path
DATA_DIR = project_root / 'data' / 'raw'

# --- Import Your Custom Ingestion Logic ---
# Now you can import from ingestion.py
from ingestion import ingest_sorption_data

# --- Run the Ingestion Pipeline ---
# This one function call processes all .xlsx files in your data directory
all_experiments = ingest_sorption_data(DATA_DIR)

# --- Explore the Ingested Data ---
# Separate successful from failed ingestions
successful_ingestions = []
failed_files = []
for df, meta in all_experiments:
    # Check if the parsing function returned an error
    if 'error' in meta:
        failed_files.append(meta.get('source_file', 'Unknown File') + f": {meta['error']}")
    else:
        # Add sample name to the dataframe for successful ingestions
        sample_name = meta.get('sample_name', meta.get('source_file', 'unknown_sample'))
        df['sample_name'] = sample_name
        successful_ingestions.append(df)

# --- Report on Ingestion Status ---
print(f"--- Ingestion Report ---")
print(f"Successfully parsed: {len(successful_ingestions)} file(s)")
print(f"Failed to parse:   {len(failed_files)} file(s)")

if failed_files:
    print("\n--- Failure Details ---")
    for reason in failed_files:
        print(f"- {reason}")

# --- Process and Display Successfully Ingested Data ---
if successful_ingestions:
    # Concatenate all individual dataframes into a single one
    master_df = pd.concat(successful_ingestions, ignore_index=True)
    
    print("\n--- Master DataFrame Head (Combined from all files) ---")
    display(master_df.head())
    
    print("\n--- Master DataFrame Info ---")
    master_df.info()
else:
    print("\nNo data was successfully ingested to create a master DataFrame.")

2026-06-25 23:04:04,790 - INFO - Starting ingestion from directory: /home/mouneer_alrayes/graphage13-sorption/data/raw
2026-06-25 23:04:04,792 - INFO - Processing file: 4M13 sample 3 17-05-24-0-90-0 0.002% stop 30min-12hr-2024-05-17 14-25-33.xlsx
2026-06-25 23:04:06,937 - INFO - Successfully parsed 2415 data points.
2026-06-25 23:04:06,939 - INFO - Processing file: GPA 0.002% 0-90-0 x 3.xlsx
2026-06-25 23:04:18,145 - INFO - Successfully parsed 14970 data points.
2026-06-25 23:04:18,146 - INFO - Processing file: GPA 80_ sample 3-0-90-0 0.002_ stop 30min-12hr-2024-06-07 13-43-07.xlsx
2026-06-25 23:04:21,666 - INFO - Successfully parsed 4517 data points.
2026-06-25 23:04:21,668 - INFO - Processing file: 0.25M13 sample 3-0-90-0 0.002% stop 30min-12hr-2024-05-31 15-04-20.xls
2026-06-25 23:04:22,508 - INFO - Successfully parsed 3391 data points.
2026-06-25 23:04:22,509 - INFO - Processing file: baked GPA sample 3-0-90-0 0.002_ stop 30min-12hr-2024-05-23 15-39-48.xls
2026-06-25 23:04:23,568 -

--- Ingestion Report ---
Successfully parsed: 5 file(s)
Failed to parse:   0 file(s)

--- Master DataFrame Head (Combined from all files) ---


,time_minutes_,mass_mg_,delta_mass_,dm_dt_minute_,target_incubator_temp_celsius_,measured_incubator_temp_celsius_,target_preheater_temp_celsius_,measured_preheater_temp_celsius_,sorption_temp_celsius_,target_partial_pressure_solvent_a_,...,unnamed_84,target_chiller_temp_celsius_,measured_chiller_temp_celsius_,measured_chiller_internal_temp_celsius_,chiller_state,gas_concentration_sensor_a_ppm_,gas_concentration_sensor_b_ppm_,sample_name,target_p_p0,dm_
0,0.00,3.7727,75.454,-394.5271,25,24.8,0,0,24.9,0.0,...,0,-10,0,0,Off,0,0,Active Reservoir:,NaN,NaN
1,1.00,3.7041,74.082,-0.4714,25,25.0,0,0,24.9,0.0,...,0,-10,0,0,Off,0,0,Active Reservoir:,NaN,NaN
2,2.01,3.6604,73.208,379.7382,25,25.0,0,0,25.0,0.0,...,0,-10,0,0,Off,0,0,Active Reservoir:,NaN,NaN
3,3.01,3.6357,72.714,709.7727,25,25.0,0,0,25.0,0.0,...,0,-10,0,0,Off,0,0,Active Reservoir:,NaN,NaN
4,4.01,3.6196,72.392,110.2624,25,25.0,0,0,25.0,0.0,...,0,-10,0,0,Off,0,0,Active Reservoir:,NaN,NaN



--- Master DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30076 entries, 0 to 30075
Data columns (total 94 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   time_minutes_                            30076 non-null  float64
 1   mass_mg_                                 30076 non-null  float64
 2   delta_mass_                              30076 non-null  float64
 3   dm_dt_minute_                            30076 non-null  float64
 4   target_incubator_temp_celsius_           30076 non-null  int64  
 5   measured_incubator_temp_celsius_         30076 non-null  float64
 6   target_preheater_temp_celsius_           30076 non-null  int64  
 7   measured_preheater_temp_celsius_         30076 non-null  int64  
 8   sorption_temp_celsius_                   30076 non-null  float64
 9   target_partial_pressure_solvent_a_       15106 non-null  float64
 10  measured_partia

In [2]:
# --- Step 1.5: Isolate Kinetic Modeling Essentials ---

# Define the precise column parameters needed for mass-uptake calculations
essential_columns = [
    'sample_name', 
    'time_minutes_', 
    'mass_mg_', 
    'delta_mass_', 
    'dm_dt_minute_', 
    'sorption_temp_celsius_',
    'measured_partial_pressure_solvent_a_'  # This tracks your relative humidity / P/P0 steps
]

# Extract only the essential features into a pristine modeling dataframe
kinetic_df = master_df[essential_columns].copy()

# Rename the primary humidity column to something standard for easier math scripting
kinetic_df.rename(columns={'measured_partial_pressure_solvent_a_': 'measured_rh'}, inplace=True)

print("--- Pristine Kinetics DataFrame Verified ---")
kinetic_df.info()

--- Pristine Kinetics DataFrame Verified ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30076 entries, 0 to 30075
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   sample_name             30076 non-null  object 
 1   time_minutes_           30076 non-null  float64
 2   mass_mg_                30076 non-null  float64
 3   delta_mass_             30076 non-null  float64
 4   dm_dt_minute_           30076 non-null  float64
 5   sorption_temp_celsius_  30076 non-null  float64
 6   measured_rh             30076 non-null  float64
dtypes: float64(6), object(1)
memory usage: 1.6+ MB
